# ECERAG on Kaggle — generator-scale sweep + improved calibration

Runs the full ECERAG experiment (Baseline RAG / ECERAG / ECERAG+CR, clean + 3 perturbations,
all 60 benchmark questions) on GPU, for several generator sizes, then analyses the results
under both the paper's original protocol and the improved cross-validated protocol.

**Notebook settings (right-hand panel) before running:**
- *Accelerator*: **GPU T4 x2** (a 7B model in fp16 is split across both T4s). On a single P100, set `LOAD_4BIT = True` below.
- *Internet*: **On** (needs a phone-verified Kaggle account) — models download from Hugging Face.
- *Persistence*: Files (optional) so a restarted session can resume.

**Expected runtime** on 2x T4: ~10 min (0.5B) · ~15 min (1.5B) · ~25 min (3B) · ~70 min (7B) · ~2 h (14B, 4-bit),
plus ~20 min judging — about 4.5 h in total, inside Kaggle's 12 h session limit.
Use **Save Version → Save & Run All** so it runs in the background and you can close the tab;
outputs (the `ecerag_results.zip`) appear under the version's *Output* tab.

`run_full.py` is resumable: if the session dies, re-run the notebook and finished questions are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = "https://github.com/yomisys/LLM"
BRANCH = "kaggle-experiment"
!git clone -q -b {BRANCH} --depth 1 {REPO} /kaggle/working/LLM 2>/dev/null || (cd /kaggle/working/LLM && git pull -q)
%cd /kaggle/working/LLM
!pip install -q rank_bm25

In [ ]:
import os
# Generators to compare. The paper used 0.5B; the sweep shows whether the
# selective-risk gains it predicted appear once generation errors stop dominating.
GENERATORS = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct",   # reviewers asked for a generator larger than 7B
]
# Models loaded in 4-bit: 14B in fp16 (~30 GB) does not fit on 2x T4 (2x 15 GB)
FOUR_BIT = {"Qwen/Qwen2.5-14B-Instruct"}
# Judge from a different model family than the generators (avoids self-preference bias).
JUDGE_MODEL = "microsoft/Phi-3.5-mini-instruct"
RUN_JUDGE = True
# On a single 16GB GPU (P100) also add the 7B model to FOUR_BIT.

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if FOUR_BIT:
    !pip install -q bitsandbytes
# Optional stronger retrieval stack (uncomment to try; builds its own embedding cache):
# os.environ["ECERAG_EMBED_MODEL"] = "BAAI/bge-base-en-v1.5"
# os.environ["ECERAG_RERANKER_MODEL"] = "BAAI/bge-reranker-base"

### 1. Smoke test (2 questions, smallest model) — fails fast if anything is misconfigured

In [ ]:
!ECERAG_GENERATOR_MODEL=Qwen/Qwen2.5-0.5B-Instruct python code/run_full.py --limit 2 --out results/kaggle_smoke
!python code/analyze.py results/kaggle_smoke > /dev/null && echo "smoke test OK"

### 2. Phase 1 — generate (GPU). One process per model so GPU memory is freed between models.

In [ ]:
for g in GENERATORS:
    print(f"===== {g}")
    q4 = "1" if g in FOUR_BIT else "0"
    !ECERAG_GENERATOR_MODEL={g} ECERAG_LOAD_4BIT={q4} python code/run_full.py

### 3. Optional — LLM-judge grading (semantic check on the lexical grader)

In [ ]:
if RUN_JUDGE:
    for g in GENERATORS:
        d = f"results/kaggle/{g.split('/')[-1]}"
        !ECERAG_JUDGE_MODEL={JUDGE_MODEL} python code/judge_answers.py {d}

### 4. Phase 2 — calibrate + evaluate (CPU, seconds)

In [ ]:
dirs = " ".join(f"results/kaggle/{g.split('/')[-1]}" for g in GENERATORS)
!python code/analyze.py {dirs} --grader lexical > /dev/null
if RUN_JUDGE:
    !python code/analyze.py {dirs} --grader judge > /dev/null

from IPython.display import Markdown, display
graders = ["lexical"] + (["judge"] if RUN_JUDGE else [])
for grader in graders:
    display(Markdown(open(f"results/kaggle/scale_sweep_{grader}.md").read()))

In [ ]:
# Full per-model report (main tables, ablation ladder, bootstrap CIs, paper-protocol replication)
for g in GENERATORS:
    d = f"results/kaggle/{g.split('/')[-1]}"
    display(Markdown(open(f"{d}/report_{graders[-1]}.md").read()))

### 5. Figures

In [ ]:
import json, re
import matplotlib.pyplot as plt

grader = graders[-1]
sizes, base_r, ec_r, base_c, ec_c = [], [], [], [], []
for g in GENERATORS:
    a = json.load(open(f"results/kaggle/{g.split('/')[-1]}/analysis_{grader}.json"))
    cv = a["cv_protocol"]
    sizes.append(float(re.search(r"([\d.]+)B", g).group(1)))
    base_r.append(cv["baseline"]["clean"]["selective_risk"]); base_c.append(cv["baseline"]["clean"]["coverage"])
    m = cv["variants"][cv["main_variant"]]["clean"]["ecerag"]
    ec_r.append(m["selective_risk"]); ec_c.append(m["coverage"])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(sizes, base_r, "o-", label="Baseline RAG"); ax[0].plot(sizes, ec_r, "s-", label="ECERAG (fitted)")
ax[0].set(xscale="log", xlabel="generator size (B params)", ylabel="selective risk (clean)", title="Error rate on answered queries")
ax[1].plot(sizes, base_c, "o-", label="Baseline RAG"); ax[1].plot(sizes, ec_c, "s-", label="ECERAG (fitted)")
ax[1].set(xscale="log", xlabel="generator size (B params)", ylabel="coverage (clean)", title="Fraction answered")
for a_ in ax: a_.legend(); a_.grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/kaggle/fig_scale_sweep.png", dpi=200); plt.show()

### 6. Human-evaluation sheet
Exports 100 answered items from the largest model (stratified by question type × condition, blind to the automated
grades) to `human_eval_sheet.csv`. Give a copy to two annotators, then score locally with
`python code/human_eval.py score results/kaggle/<model> annot_A.csv annot_B.csv`.

In [ ]:
!python code/human_eval.py export results/kaggle/{GENERATORS[-1].split('/')[-1]} --n 100

### 7. Download results

In [ ]:
!cd results && zip -qr /kaggle/working/ecerag_results.zip kaggle && ls -lh /kaggle/working/ecerag_results.zip